# Global Superstore Sales Analysis: 04. Exploratory Data Analysis (EDA)
**Author:** Sneh Kheni  
**Role:** Senior Data Analyst & Analytics Engineer  
**Objective:** Conduct in-depth exploratory data analysis to evaluate product profitability, regional performance, customer concentration, and the empirical impact of discounts.


In [1]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image

sys.path.insert(0, os.path.abspath(".."))
from scripts.eda_visualizer import (
    plot_category_performance,
    plot_sub_category_profitability,
    plot_regional_performance,
    plot_discount_elasticity,
    plot_top_bottom_products,
    plot_customer_segments,
    plot_monthly_timeline
)

data_path = "../data/processed/cleaned_superstore.csv" if os.path.exists("../data/processed/cleaned_superstore.csv") else "data/processed/cleaned_superstore.csv"
output_dir = "../outputs/visuals" if os.path.exists("../outputs/visuals") else "outputs/visuals"
df = pd.read_csv(data_path)
print("Data loaded for EDA. Generating visual figures...")


## 1. Product Category & Sub-Category Profitability
Identify profit drivers and structural loss-makers across the 17 sub-categories.


In [2]:
sub_cat = df.groupby(["Category", "Sub.Category"]).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Order_Count=("Order.ID", "count"),
    Avg_Discount=("Discount", "mean")
).reset_index()
sub_cat["Profit_Margin_%"] = (sub_cat["Total_Profit"] / sub_cat["Total_Sales"] * 100).round(2)
sub_cat.sort_values(by="Total_Profit", ascending=False)


In [3]:
img_path = plot_sub_category_profitability(df, output_dir)
Image(filename=img_path)


## 2. Global Regional & Market Performance


In [4]:
img_path = plot_regional_performance(df, output_dir)
Image(filename=img_path)


## 3. The Discount Trap: Discount Rate vs. Profitability
Analyze empirical margin erosion across discount tiers.


In [5]:
disc_analysis = df.groupby("Discount_Band", observed=False).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Order_Count=("Order.ID", "count"),
    Loss_Orders=("Profit", lambda x: (x < 0).sum())
).reset_index()
disc_analysis["Profit_Margin_%"] = (disc_analysis["Total_Profit"] / disc_analysis["Total_Sales"] * 100).round(2)
disc_analysis["Loss_Rate_%"] = (disc_analysis["Loss_Orders"] / disc_analysis["Order_Count"] * 100).round(2)
disc_analysis


In [6]:
img_path = plot_discount_elasticity(df, output_dir)
Image(filename=img_path)


## 4. Top 10 Profitable Products vs. Top 10 Loss-Making Products


In [7]:
img_path = plot_top_bottom_products(df, output_dir)
Image(filename=img_path)


## 5. Customer Segment Breakdown


In [8]:
img_path = plot_customer_segments(df, output_dir)
Image(filename=img_path)


## Key Findings
1. **Severe Loss Maker:** Tables sub-category generated -$64,083 in losses despite $757K in sales, driven by an average discount rate exceeding 28%.
2. **Profit Champions:** Copiers ($258.5K profit, 17.1% margin) and Phones ($216.7K profit, 12.7% margin) dominate overall profitability.
3. **Discount Cliff:** At discounts above 20%, average profit margin plummets from +28% (no discount) to -17% (21-30% discount) and -88% (>50% discount).
4. **Regional Disparity:** Central ($311K profit) and North Asia ($166K profit) outperform, while Southeast Asia Furniture operates at a net loss (-$7.3K).
